# 03 - Embeddings
## Shakespeare Evidence Desk


## Why This Stage

*In general — what embeddings are and why any RAG system needs them.* Literal keyword matching breaks the moment a question and its answer use different words — synonyms, paraphrases, and related concepts slip straight through it. Embeddings solve this by turning each piece of text into a point in a high-dimensional space, positioned so that *semantic similarity* — meaning, not wording — decides what sits nearby. Search then becomes geometry: find the points closest to the query's point. This is the stage where every unit of text receives the numeric representation that all of retrieval will later search over.

Embeddings come after chunking (you can only encode the units you have already decided on) and before indexing and retrieval (which need vectors to organize and search). The representation chosen here is a hard ceiling on everything downstream: if two unrelated passages land close together in this space, no index or reranker can reliably separate them afterward. That is why the embedding is treated as a measured choice — which model, how many dimensions, whether vectors are normalized — rather than an invisible default.


- **Goal:** turn each frozen chunk from notebook 02 into a dense vector that places semantically similar passages near each other, so retrieval (notebook 05) can find evidence by meaning rather than exact words.
- **Loads from Step 02:** `.artifacts/chunks.json` (the selected chunks with full provenance) and `.artifacts/chunking-handoff.json` (the chunking decision). The labeled questions (`.artifacts/eval-questions.json`) drive the retrieval-quality proxy.
- **The experiment:** hold the chunking fixed and vary the one knob the deployed embedding model exposes — the **output dimension** — comparing truncated 256 / 512 / 1024 against the model's **full native dimension**, plus a **normalization** decision (store unit vectors or not) and the **batching + caching** mechanics that make embedding affordable.
- **One deployment, honest scope:** we have a single embedding deployment, so this notebook does not compare separate model families (that needs more deployments — noted as future work). It compares the dimension variants the 3-series model produces natively via the *Matryoshka* property.
- **Offline-safe:** embedding is a network stage, but if Azure is unavailable the notebook falls back to a deterministic local pseudo-embedding so the full pipeline still runs — clearly flagged as **not** meaningful for the actual selection.
- **Not later stages:** this notebook does not build an ANN index, retrieve, rerank, or answer. It exports vectors and an embedding decision for notebook 04 (indexing).


## Conceptual Foundations

### From words to coordinates

An **embedding** is a function that maps a piece of text to a fixed-length list of numbers — a **vector** — positioned in a high-dimensional space so that *meaning* determines location. Texts about similar things land near each other; unrelated texts land far apart. The embedding model learned this geometry from enormous amounts of text, so "the king was poisoned" and "the monarch was given poison" end up close even though they share almost no words. This is what lets retrieval match by *meaning* rather than exact wording — the problem keyword search cannot solve.

### Why "search" becomes "geometry"

Once every chunk is a point, finding relevant chunks for a query becomes: embed the query into the *same* space, then find the nearest points. "Nearest" needs a **distance metric**:

- **Cosine similarity** — the angle between two vectors; ignores length, measures only *direction* (which is where meaning lives). The default for text embeddings.
- **Dot (inner) product** — multiply matching coordinates and sum. For **unit-length** vectors (length 1), dot product *equals* cosine — so if you normalize, you get cosine's quality with the cheaper, index-friendly operation. This equivalence is exactly why this notebook stores normalized vectors.
- **Euclidean (L2) distance** — straight-line distance; for normalized vectors it ranks identically to cosine.

### The dimension knob and the Matryoshka property

A vector's **dimension** is how many numbers describe each text (e.g. 1536, or a shorter prefix like 512). More dimensions can capture finer distinctions but cost more storage and search time. Modern OpenAI 3-series models are trained with a **Matryoshka** property: the *first* N numbers of a full vector form a valid, slightly coarser embedding on their own. That means you can embed once at full width and then **truncate + renormalize** to 1024/512/256 locally — no re-embedding — which is the free experiment this notebook runs.

### What this stage can and cannot change

Embedding is the quality **ceiling** for everything downstream: if two unrelated passages land close here, no index (04) or reranker (06) can reliably separate them afterward. But this stage has limited knobs because we have a *single* deployed model. The biggest lever — *which model* — is discussed but not experimented on (it needs more deployments). What we *can* tune fairly is the output dimension, plus settled mechanics (normalization, batching, caching).

### Common misconceptions

- **"More dimensions are always better."** Beyond a point, extra dimensions add storage and latency for negligible retrieval gain; the right width is the smallest that holds quality.
- **"Normalization changes the results."** It changes the *math used*, not the *ranking* — dot-on-normalized equals cosine-on-raw (the notebook verifies this). It's a free efficiency win.
- **"A good embedding guarantees good retrieval."** It sets the ceiling, but chunking, indexing, and reranking all operate on top of it and can still lose the right passage.
- **"Embeddings understand the text."** They encode statistical meaning learned from data; they don't reason. They make *similar* things *close*, nothing more — which is enough for retrieval.


## 1. Load the Previous Selection

- **Purpose:** start from notebook 02's exported chunks and decision record, not hidden kernel state. Each chunk already carries its text and provenance (`work/act/scene/speakers/lines`), so this stage only has to encode text — the structure is fixed.
- **What the code does:** locates the repository root (using `.env.example` as a marker), reads `chunks.json`, validates that every chunk still carries the provenance fields retrieval depends on, groups chunks by work, and loads the labeled questions the quality proxy (Step 4) will use.
- **Fails loudly if missing:** if `.artifacts/chunks.json` is absent, run notebook 02 first — this notebook cannot invent the chunk set.
- **Expected result:** the selected chunking strategy, the chunk count across the ten plays, the total token volume to embed (a cost signal), and the development / held-out question counts. No network call happens here.


In [ ]:
import json
import re
from collections import defaultdict
from pathlib import Path

# Work from the repository root so the notebook and the package read the same .env and artifacts.
ROOT = Path.cwd().resolve()
if not (ROOT / '.env.example').is_file() and (ROOT.parent / '.env.example').is_file():
    ROOT = ROOT.parent
if not (ROOT / '.env.example').is_file():
    raise RuntimeError('Use the repository as the notebook working directory.')

ARTIFACTS = ROOT / '.artifacts'
chunks_path = ARTIFACTS / 'chunks.json'
chunking_handoff_path = ARTIFACTS / 'chunking-handoff.json'
questions_path = ARTIFACTS / 'eval-questions.json'
if not chunks_path.is_file():
    raise RuntimeError('Missing .artifacts/chunks.json. Run notebook 02 first.')

chunks_data = json.loads(chunks_path.read_text(encoding='utf-8'))
# Notebook 02 writes {corpus_prefix, strategy, chunks: [...]}; accept a bare list too for forward compatibility.
chunks = chunks_data['chunks'] if isinstance(chunks_data, dict) else chunks_data
chunking_handoff = json.loads(chunking_handoff_path.read_text(encoding='utf-8')) if chunking_handoff_path.is_file() else {}
evaluation_questions = json.loads(questions_path.read_text(encoding='utf-8')) if questions_path.is_file() else {'development': [], 'heldout': []}

# Fail now if a chunk lost the provenance fields that retrieval and citations depend on downstream.
required_fields = {'chunk_id', 'work_id', 'work', 'scenes', 'text', 'token_count'}
missing = [c.get('chunk_id', '?') for c in chunks if not required_fields.issubset(c)]
if missing:
    raise RuntimeError(f'{len(missing)} chunk(s) missing provenance fields; re-run notebook 02.')

# Prefer the handoff's record of the winning strategy; fall back to the wrapper's own label.
selected_strategy = chunking_handoff.get('selected_strategy') or (
    chunks_data.get('strategy') if isinstance(chunks_data, dict) else None) or 'unknown'
work_titles = {c['work_id']: c['work'] for c in chunks}
chunks_by_work = defaultdict(list)
for c in chunks:
    chunks_by_work[c['work_id']].append(c)

total_tokens = sum(c['token_count'] for c in chunks)
print(f'Loaded {len(chunks):,} chunks from strategy "{selected_strategy}" across {len(chunks_by_work)} works.')
print(f'Total chunk tokens to embed: {total_tokens:,}')
print(f'Questions: {len(evaluation_questions.get("development", []))} development, '
      f'{len(evaluation_questions.get("heldout", []))} held-out.')


## 2. Explain the Alternatives

An embedding turns a passage into a point in space, so that passages about similar things land near each other and "search" becomes "find the nearest points." A choice is **good** when it keeps the right scene near the question while costing as little storage and search time as possible. We **fix the model** (we only have one embedding deployment) and tune the knobs that model still lets us turn.

**The choices, in easy terms:**

| Choice | What it means (plain) | Why it matters |
|---|---|---|
| **Dimension** *(our main experiment)* | How many numbers describe each passage — the full width (e.g. 1536) or a shorter prefix like 1024 / 512 / 256. These models are trained so a prefix is still a valid, slightly coarser description. | Fewer numbers = a smaller, faster index, but cut too far and nearby scenes blur together and the wrong one is retrieved. We look for the smallest width that still finds the right scene. |
| **Normalization** | Rescale every vector to the same length (length 1). | It changes nothing about *which* passages rank first, but it lets the next stage (indexing) use a faster kind of math. So: always do it. |
| **Input construction** | *What text we actually hand the model* — just the bare chunk, or the chunk with a little context in front of it (`"Hamlet — Act 1, Scene 5, Hamlet: ..."`). | The model can only encode what it sees; adding the play/act/scene/speaker can make a passage easier to match to a question. A real quality lever, and free to try. |
| **Batching** | Send many passages per request instead of one at a time. | Pure speed and cost — many-per-call is far cheaper and faster. No effect on quality. |
| **Caching** | Save each vector to disk keyed by its text, so re-runs don't re-embed. | The embedding pass is the expensive part; caching makes re-tuning this stage essentially free. No effect on quality. |

- **Why dimension is the experiment we actually run:** it is the one quality knob we can test fairly and cheaply with a single model — embed once at full width, then shorten and re-measure. Normalization, batching, and caching are settled decisions, not experiments.
- **Honest expectation:** there is no free "best vector." We measure a retrieval proxy (does the right scene rank near the top for a labeled question?), storage, and cost, then pick the **smallest dimension that keeps quality within tolerance** of the full width — the simplest choice that meets the constraint.

> **Footnote — why comparing *models* matters (even though we don't here).** The single biggest lever at this stage is usually **which embedding model you use** — a stronger or domain-tuned model (or `text-embedding-3-large` vs `-small`, or a non-OpenAI encoder) can lift retrieval more than any dimension tweak. We don't test it because we have only **one deployment**, and comparing models fairly means standing up each one, re-embedding the whole corpus, and re-running the same retrieval proxy on identical questions. It is deferred, not forgotten: when a second embedding deployment exists, model choice should be the **first** thing re-measured, because everything downstream (indexing, retrieval, generation quality) inherits the ceiling the embedding model sets.

### Where each choice lives in the code

A map from the choices above to the functions that implement and measure them (named by section and function, so it stays valid if cells move):

| Choice | Implemented by | Measured / checked by | Status |
|---|---|---|---|
| **Dimension** | `truncate_dims` builds each variant (`chunk_reps`) — Section 3 | `evaluate_rep` + `dot` score each (`dim_metrics`); `within_tolerance` picks the winner — Sections 4–5 | implemented **+ measured + decided** |
| **Normalization** | `l2_normalize` (inside `truncate_dims`) — Section 3 | the `norm_top == raw_top` check — Section 4 | implemented **+ checked safe** |
| **Batching** | the `EMBED_BATCH` loop in `embed_texts` → `_embed_raw` — Section 3 | `usage_log['api_calls']` printout — Section 3 | implemented, cost observed |
| **Caching** | `_cache_key` + the read/write in `embed_texts` — Section 3 | the "0 API calls" on a re-run — Section 3 | implemented, effect observed |
| **Input construction** | — none (bare `c['text']` is embedded) | — none | **gap: described, not yet run** |

The **answer key** that every quality score is graded against is built by `roman_to_int` → `resolve_targets` → `chunk_covers` (the `gold` list, Section 4). So **4 of 5** choices have code; input construction is the one described here but not yet implemented or measured.


> ### Footnote — encoder architectures: bi-encoder, cross-encoder, ColBERT
>
> Background on *the kind of model* doing the embedding. It is **conceptual grounding**, not part of the experiment — skim it the first time, reread it when the terms blur. The one word for the model that turns text into vectors is an **encoder**; the three designs below differ in *when* the query and the passage meet.
>
> **Q: What is this notebook actually using?** A **bi-encoder** (a.k.a. *dual encoder*). The query and every passage are encoded **separately and independently** into one vector each, and relevance is a single `dot(query_vec, passage_vec)`. That independence is the whole reason indexing works: passages are encoded **once, offline** (this notebook) and frozen into an index (notebook 04); at query time only the query is encoded, then compared against the stored vectors. Fast and scalable, but it compresses a whole passage into one point, so fine word-level detail can blur.
>
> **Q: What is a cross-encoder, then?** The opposite tradeoff. It feeds the query and a passage **together** into the model and outputs a single relevance score — so it reads them *jointly* and is far more accurate. The cost: nothing can be pre-computed (every query–passage pair is a fresh forward pass), so it **cannot be indexed** and only runs over a small candidate set. That is exactly the **reranker in notebook 06**: dense bi-encoder retrieval casts a wide, cheap net; a cross-encoder re-scores the top handful.
>
> **Q: Where does ColBERT fit?** In between, via **late interaction**. Instead of one vector per passage, ColBERT stores **one vector per token** and scores with **MaxSim**: for each query token, take its best-matching passage token, then sum those maxima.
>
> $$\text{score}(q,d) = \sum_{i \in q} \max_{j \in d} \; \mathbf{E}_{q_i} \cdot \mathbf{E}_{d_j}$$
>
> It keeps word-level precision (great for exact quotes and rare names — a specific Shakespeare line) while **still precomputing passage vectors offline**, so it scales far better than a cross-encoder. The price is storage and a specialized index: many vectors per passage instead of one.
>
> | Architecture | Vectors stored per passage | Query & passage meet… | Scoring | Scales to index? | In this repo |
> |---|---|---|---|---|---|
> | **Bi-encoder** *(dense)* | **1** | never (separately encoded) | one dot product | ✅ yes (offline) | **used here** — nb 03 passages, nb 04 index, nb 05 dense retrieval |
> | **ColBERT** *(late interaction)* | **many** (one per token) | late, per-token (MaxSim) | sum of per-token maxima | ⚠️ yes, but heavy index | not used (natural upgrade between dense and rerank) |
> | **Cross-encoder** | **0** (nothing stored) | early (read jointly) | single joint score | ❌ no (per-query only) | **nb 06** reranking |
>
> **Q: So why is the bi-encoder the right default here?** Because an index needs vectors it can compute *before* the query exists — only the bi-encoder (and, more heavily, ColBERT) can do that. The bi-encoder sets the recall ceiling cheaply; the cross-encoder reranker (nb 06) then sharpens the top of the list where joint reading pays off. ColBERT is the middle option to reach for if single-vector recall proves too coarse before paying for a reranker.


## The Embedding Landscape: Techniques, Tools, and How to Choose

The encoder footnote above covered *architectures* (how a model is shaped). This section covers the *menu of embedding approaches and models* you choose from in practice, and how to decide.

Legend: **[IMPLEMENTED]** used here · **[EXPLAINED]** relevant, not used · **[ADVANCED]** specialized.

### Families of representation

| Family | What the vector is | When it shines | Status |
|---|---|---|---|
| **Dense embeddings** | one fixed-length vector of ~hundreds–thousands of floats, capturing meaning | semantic/paraphrase matching (the RAG default) | **[IMPLEMENTED]** |
| **Sparse embeddings** (BM25, SPLADE) | a huge mostly-zero vector over the vocabulary, capturing exact terms | exact keywords, rare names, codes, quotes | **[EXPLAINED]** (hybrid search, notebook 05) |
| **Multi-vector / late interaction** (ColBERT) | one vector *per token* | word-level precision at scale | **[EXPLAINED]** (encoder footnote) |
| **Instruction-tuned embeddings** | dense, but conditioned on a task prompt ("represent this for retrieval:") | mixed query/document asymmetry | **[ADVANCED]** |

**Key takeaway:** dense and sparse are complementary, not competing. Dense finds *meaning*; sparse finds *exact strings*. Production retrieval often fuses both (**hybrid search**) — which is why this notebook builds a dense index now and notebook 05 adds the keyword/BM25 half.

### Choosing a model (the biggest lever, which we hold fixed)

Common options you should know, and what differentiates them:

- **Azure OpenAI / OpenAI `text-embedding-3-small` / `-large`** (our model) — strong general quality, Matryoshka-truncatable dimensions, hosted, pay-per-token.
- **Cohere Embed, Voyage AI** — competitive hosted models, often with retrieval- and domain-tuned variants.
- **Open-weight: BGE, E5, GTE, Nomic, Jina** — run locally via `sentence-transformers`; free at inference, private/offline, but you host the compute.
- **Multilingual / domain-specific variants** — when your corpus isn't English or is specialized (legal, biomedical, code).

**How to choose:**

```
Need offline / on-prem / zero per-token cost?   → open-weight (BGE/E5/Nomic) via sentence-transformers
Want best-in-class quality with no infra?       → hosted (OpenAI 3-large, Cohere, Voyage)
Corpus is multilingual or domain-specific?      → a matching multilingual/domain model
Tight storage/latency budget at scale?          → smaller dimension or a Matryoshka-truncatable model
```

Use the **MTEB leaderboard** (Massive Text Embedding Benchmark) as a starting shortlist, then **re-measure on your own labeled questions** — leaderboard rank rarely transfers perfectly to a specific corpus. This project cannot compare model *families* (one deployment), so it measures the one knob it can vary fairly: the output dimension.

### Tools and libraries

- **`openai` / Azure OpenAI SDK** — our client for the hosted embedding deployment.
- **`sentence-transformers`** — the standard way to run open-weight encoders (bi-encoders, cross-encoders, ColBERT) locally.
- **`tiktoken`** — token counting, to stay within the model's input window.
- **MTEB** — benchmark + leaderboard for shortlisting models.
- **Vector DB client libraries** (notebook 04) — most accept whatever vectors you produce here, so the embedding choice and the index choice are largely independent.

### Parameters and mechanics settled (not experimented) here

- **Normalization = on** — store unit vectors so inner product equals cosine; verified safe, enables fast index math.
- **Batching (128/call)** — pure cost/latency win, no quality effect.
- **Caching (keyed by deployment + text)** — makes the first run cheap and every re-run free; the cache key includes the model so switching models never serves a stale vector.


## 3. Run Controlled Experiments

- **Fixed upstream:** the chunk set, the embedding deployment, and the query set are held constant; only the output dimension changes between candidates.
- **What the code does:** connects to the hosted embedding deployment (keyless, like notebooks 00–02), embeds every chunk **once** at the model's full native dimension with **batching** and on-disk **caching**, then derives the 1024 / 512 / 256 candidates locally by **truncate + L2-normalize** (the Matryoshka property) so no chunk is embedded more than once.
- **Offline-safe:** if `.env` or Azure credentials are unavailable, a deterministic hash-based pseudo-embedding stands in so batching, normalization, and the retrieval proxy still execute end to end. Its vectors are **not** meaningful — the markdown and output say so, and the selection must be re-run against the real backend.
- **Expected result:** the backend in use, the native dimension discovered, the number of API calls and tokens spent (cost signal), and the list of candidate dimensions to compare.

> **Note — what one run actually costs.** The *first* run embeds all chunks once at full width (a handful of batched API calls) and writes every vector to `.artifacts/embedding-cache/`. The 256 / 512 / 1024 candidates are then made *locally* by truncating that one full-width vector — **no extra API calls**. Every later run reads the cache and reports **0 API calls**, so re-measuring, changing the tolerance, or changing the selection rule is instant and free. A run is deterministic given the same chunks and model, so numbers are reproducible across runs.


In [ ]:
import hashlib
import math
import time

EMBED_BATCH = 128                  # texts per API call: amortizes overhead while staying under row/token limits
CANDIDATE_DIMS = [256, 512, 1024]  # Matryoshka prefixes compared against the full native dimension
CACHE_DIR = ARTIFACTS / 'embedding-cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def l2_normalize(vector):
    # Make a vector length 1 so later a plain dot product equals cosine similarity.
    norm = math.sqrt(sum(value * value for value in vector))
    return [value / norm for value in vector] if norm else vector  # leave an all-zero vector untouched


def truncate_dims(vector, dims):
    # Build a smaller vector by keeping the first `dims` numbers and renormalizing (the Matryoshka property),
    # so every dimension candidate comes from one full-width embedding with no re-embedding.
    return l2_normalize(vector[:dims])


# ---- connect to the hosted embedding deployment (keyless, like notebooks 00-02) ----
embedding_backend = {'available': False, 'deployment': None, 'native_dims': None}
_client = None
_deployment = None
try:
    from dotenv import dotenv_values
    from azure.identity import DefaultAzureCredential, get_bearer_token_provider
    from openai import AzureOpenAI

    config = {key: value for key, value in dotenv_values(ROOT / '.env').items() if value}
    token_provider = get_bearer_token_provider(
        DefaultAzureCredential(), 'https://cognitiveservices.azure.com/.default',
    )
    _client = AzureOpenAI(
        azure_endpoint=config['AZURE_OPENAI_ENDPOINT'],
        api_version=config['AZURE_OPENAI_API_VERSION'],
        azure_ad_token_provider=token_provider,
    )
    _deployment = config['AZURE_OPENAI_EMBEDDING_DEPLOYMENT']
    # Probe once so bad credentials fail here (caught below) instead of mid-run.
    _client.embeddings.create(model=_deployment, input=['probe'])
    embedding_backend.update({'available': True, 'deployment': _deployment})
except Exception as error:
    # Offline-safe: fall back to local pseudo-vectors so the pipeline still runs end to end.
    print(f'Hosted embeddings unavailable ({type(error).__name__}); using a deterministic offline fallback.')
    print('Fallback vectors are NOT meaningful — do not trust the selection until re-run against Azure.')

FALLBACK_DIMS = 1536


def _fallback_embedding(text):
    # Deterministic hash-derived vector, used only when Azure is unavailable; not semantically meaningful.
    out, counter = [], 0
    seed = text.encode('utf-8')
    while len(out) < FALLBACK_DIMS:
        digest = hashlib.sha256(seed + counter.to_bytes(4, 'big')).digest()
        out.extend(byte / 255.0 - 0.5 for byte in digest)
        counter += 1
    return out[:FALLBACK_DIMS]


def _cache_key(text):
    # Key cached vectors by (deployment, text) so switching models never reuses the wrong vector.
    tag = (_deployment or 'offline').encode('utf-8')
    return hashlib.sha256(tag + b'|' + text.encode('utf-8')).hexdigest()


usage_log = {'api_calls': 0, 'total_tokens': 0, 'latency_ms': 0.0}


def _embed_raw(texts):
    # The only place that actually calls the model (or the fallback); records tokens + latency.
    if not embedding_backend['available']:
        return [_fallback_embedding(text) for text in texts]
    started = time.perf_counter()
    response = _client.embeddings.create(model=_deployment, input=texts)
    usage_log['latency_ms'] += (time.perf_counter() - started) * 1000
    usage_log['api_calls'] += 1
    usage_log['total_tokens'] += response.usage.total_tokens
    return [item.embedding for item in response.data]


def embed_texts(texts):
    # Public entry point: serve cached vectors from disk, batch only the misses, cache the results.
    # This is what makes the first run cheap and every later run free.
    results = [None] * len(texts)
    pending_idx, pending_text = [], []
    for i, text in enumerate(texts):
        cache_file = CACHE_DIR / f'{_cache_key(text)}.json'
        if cache_file.is_file():
            results[i] = json.loads(cache_file.read_text())
        else:
            pending_idx.append(i)
            pending_text.append(text)
    for start in range(0, len(pending_text), EMBED_BATCH):
        batch = pending_text[start:start + EMBED_BATCH]
        for offset, vector in enumerate(_embed_raw(batch)):
            i = pending_idx[start + offset]
            results[i] = vector
            (CACHE_DIR / f'{_cache_key(texts[i])}.json').write_text(json.dumps(vector))
    return results


# Embed every chunk once at full width (cached); derive the smaller candidates locally by truncation.
chunk_vectors = embed_texts([c['text'] for c in chunks])
native_dims = len(chunk_vectors[0])
embedding_backend['native_dims'] = native_dims

candidate_dims = [d for d in CANDIDATE_DIMS if d < native_dims] + [native_dims]
chunk_reps = {dims: [truncate_dims(v, dims) for v in chunk_vectors] for dims in candidate_dims}

backend_label = f'hosted {embedding_backend["deployment"]}' if embedding_backend['available'] else 'offline fallback'
print(f'Backend: {backend_label}; native dimension {native_dims}.')
print(f'Embedded {len(chunks):,} chunks in {usage_log["api_calls"]} API call(s), '
      f'{usage_log["total_tokens"]:,} tokens, {usage_log["latency_ms"]:.0f} ms.')
print(f'Candidate dimensions: {candidate_dims}')


## 4. Measure and Inspect

- **Four lenses:** retrieval quality (the one that matters), storage, latency/cost, and a normalization sanity check.
- **Retrieval-quality proxy — a real nearest-neighbor search, not a metadata lookup.** Each answerable labeled question cites *where* its answer lives (e.g. "Hamlet, Act 1, Scene 5"). We parse that citation to find the **gold chunks** (the ones carrying that `(work, act, scene)`), then embed the **question text only** (never the expected evidence, to avoid leakage), score it against **every chunk** by cosine similarity, and rank. We report **Recall@1** (a gold chunk ranks first), **Recall@5** (a gold chunk is in the top five), and **MRR** (mean reciprocal rank — rewards putting a gold chunk as high as possible). This measures the embedding, holding chunking fixed.
- **Storage:** bytes per vector = dimension × 4 (float32) × chunk count. Halving the dimension halves the index RAM and roughly doubles search throughput — the whole reason to try smaller vectors.
- **Latency and cost:** we report measured embedding latency and the exact token count the API billed; corpus cost scales with tokens, so the number here is the real signal (the per-million rate depends on the deployed model).
- **Normalization sanity check:** we confirm that ranking by dot product on normalized vectors is **identical** to ranking by cosine on the raw vectors for a sample query — justifying storing unit vectors so notebook 04 can use fast inner-product search.
- **Honest limit:** this proxy uses only the labeled dev questions that resolve to a parsed scene, so the sample is small. It pre-screens dimensions; the end-to-end judges are retrieval (notebook 05) and continuous evaluation (notebook 10).

> **Footnote — "search" and "measure" in plain terms (a 2-number map).** Forget 1536 numbers; pretend each passage gets just **two** coordinates: `[how-much-about-death, how-much-about-love]`. The model places passages so similar ones land near each other:
> ```
> A "the king was poisoned"     -> [0.9, 0.1]   (death)
> B "two lovers meet at night"  -> [0.1, 0.9]   (love)
> C "a feast is prepared"       -> [0.4, 0.3]   (neither)
> ```
> **SEARCH** = give the question coordinates too (*"How was the king killed?"* → `[0.85, 0.15]`), then score it against every passage with a **dot product** (multiply matching numbers, add them up) and sort:
> ```
> A: 0.85*0.9 + 0.15*0.1 = 0.78   ← highest, so A ranks #1
> C: 0.85*0.4 + 0.15*0.3 = 0.39
> B: 0.85*0.1 + 0.15*0.9 = 0.22
> ```
> The engine never "understands" the question — it just notices the question's numbers point the same way as A's numbers, because both are about death. Real embeddings do exactly this with 512 numbers instead of 2.
>
> **MEASURE** = separately, the label already tells us the correct passage is **A**. We look at where A landed: it's #1 → **Recall@1 ✅, Recall@5 ✅, MRR = 1/1 = 1.00**. A worse embedding that ranked A third would score **MRR = 1/3 = 0.33** — a lower grade. So **search finds (question → ranked list); measure grades the finding (where did the known-correct passage land?).** They are two separate steps that run back-to-back inside `evaluate_rep`.


In [ ]:
_ROMAN = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}


def roman_to_int(value):
    # Normalize scene numbers ('V' or '5' -> 5) so a label and a chunk's metadata compare equal.
    if value is None:
        return None
    text = str(value).strip().upper()
    if text.isdigit():
        return int(text)
    total, prev = 0, 0  # subtractive numerals: a smaller symbol before a larger one subtracts (IV = 4)
    for char in reversed(text):
        if char not in _ROMAN:
            return None
        current = _ROMAN[char]
        total += -current if current < prev else current
        prev = max(prev, current)
    return total or None


title_by_text = {work_id: title.lower() for work_id, title in work_titles.items()}


def resolve_targets(question):
    # Read the label's citation -> (play, act, scene) coordinates for where the answer lives.
    # Uses the label only, never the question's words — otherwise the test would leak the answer.
    text = (question.get('question', '') + ' ' + question.get('expected_evidence', '')).lower()
    works = [work_id for work_id, title in title_by_text.items() if title in text]
    pairs = re.findall(r'act\s+(\d+|[ivxlc]+)\s*,?\s*scene\s+(\d+|[ivxlc]+)',
                       question.get('expected_evidence', '').lower())
    acts_scenes = [(roman_to_int(a), roman_to_int(s)) for a, s in pairs]
    return [(work_id, act, scene) for work_id in works for act, scene in acts_scenes]


def chunk_covers(chunk, work_id, act, scene):
    # Does this chunk sit in that play + scene? Used to find which chunks ARE the correct answer.
    return chunk['work_id'] == work_id and any(a == act and s == scene for a, s in chunk['scenes'])


# THE ANSWER KEY: for each answerable question, the set of chunk indices holding its cited scene.
gold = []
for question in evaluation_questions.get('development', []):
    if not question.get('answerable'):
        continue
    targets = [(w, a, s) for w, a, s in resolve_targets(question) if a is not None and s is not None]
    gold_idx = {i for i, c in enumerate(chunks) for (w, a, s) in targets if chunk_covers(c, w, a, s)}
    if gold_idx:
        gold.append((question, gold_idx))

K = 5  # the cutoff for Recall@K


def dot(a, b):
    # Similarity between unit vectors (equals cosine) — this is the actual search score; higher = more relevant.
    return sum(x * y for x, y in zip(a, b))


def evaluate_rep(reps):
    # Grade one vector size: search each question against all chunks, then score where the correct chunk ranked.
    dims = len(reps[0])
    hits1, hits_k, reciprocal = 0, 0, 0.0
    for question, gold_idx in gold:
        query_vec = truncate_dims(embed_texts([question['question']])[0], dims)
        ranked = sorted(range(len(reps)), key=lambda i: dot(query_vec, reps[i]), reverse=True)  # search: rank all chunks
        rank = next((r for r, i in enumerate(ranked) if i in gold_idx), None)                   # where the right one landed
        if rank is not None:
            hits1 += rank == 0
            hits_k += rank < K
            reciprocal += 1.0 / (rank + 1)  # 1 for rank 0, 1/2 for rank 1, ... (mean reciprocal rank)
    n = max(len(gold), 1)  # avoid divide-by-zero when no question resolved
    return {'dims': dims, 'recall@1': hits1 / n, f'recall@{K}': hits_k / n, 'mrr': reciprocal / n}


dim_metrics = {dims: evaluate_rep(reps) for dims, reps in chunk_reps.items()}

print(f'Retrieval proxy over {len(gold)} resolved dev question(s); searching all {len(chunks):,} chunks.\n')
header = ('dims', 'bytes/vec', 'index MB', 'recall@1', f'recall@{K}', 'mrr')
print('{:>6} {:>10} {:>9} {:>9} {:>9} {:>7}'.format(*header))
for dims in candidate_dims:
    m = dim_metrics[dims]
    bytes_per_vec = dims * 4  # float32
    index_mb = bytes_per_vec * len(chunks) / (1024 * 1024)
    print('{:>6} {:>10} {:>9.2f} {:>8.0%} {:>8.0%} {:>7.3f}'.format(
        dims, bytes_per_vec, index_mb, m['recall@1'], m[f'recall@{K}'], m['mrr']))

print(f'\nEmbedding spend so far: {usage_log["total_tokens"]:,} tokens across '
      f'{usage_log["api_calls"]} API call(s), {usage_log["latency_ms"]:.0f} ms total.')

# Normalization sanity check: fast dot(normalized) must rank the same as cosine(raw), else normalizing isn't safe.
if gold:
    normalized_q = truncate_dims(embed_texts([gold[0][0]['question']])[0], native_dims)
    raw_q = embed_texts([gold[0][0]['question']])[0]

    def cosine(a, b):
        return dot(a, b) / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)) or 1.0)

    norm_top = sorted(range(len(chunks)), key=lambda i: dot(normalized_q, chunk_reps[native_dims][i]), reverse=True)[:K]
    raw_top = sorted(range(len(chunks)), key=lambda i: cosine(raw_q, chunk_vectors[i]), reverse=True)[:K]
    print(f'Normalization check: dot(normalized) vs cosine(raw) top-{K} identical? {norm_top == raw_top}')


## 5. Justify the Selection

- **Decision rule (data-driven, deterministic):** take the full native dimension as the quality baseline, then prefer the **smallest** dimension whose retrieval proxy stays **within tolerance** of that baseline — specifically within `0.02` on both MRR and Recall@5. Smaller vectors are cheaper to store and faster to search, so we only pay for width that buys measurable recall.
- **Why smallest-within-tolerance, not highest-recall:** picking the absolute best score would almost always choose the full dimension and quietly over-provision the index. The constraint we actually have downstream (notebook 04) is index size and search latency; this rule honors it while refusing to trade away real retrieval quality.
- **Normalization is settled separately:** we store **unit-length** vectors because the sanity check showed dot-product ranking on normalized vectors is identical to cosine on raw vectors — so notebook 04 gets fast inner-product search for free, with no ranking change.
- **Offline caveat is explicit:** if this ran on the deterministic fallback, the "winner" is mechanical only. The printed record flags the backend so a fallback selection is never mistaken for a real one.
- **Expected result:** the selected dimension with the concrete numbers that justify it (quality within tolerance, storage saved), the normalization decision, and the rejected dimensions noted with their measured gap.


In [ ]:
TOLERANCE = 0.02   # max acceptable drop vs the full-width baseline, on both MRR and Recall@K
normalize = True   # store unit vectors so indexing can use fast dot-product search (== cosine)

baseline_dims = native_dims
baseline = dim_metrics[baseline_dims]


def within_tolerance(metrics):
    # True if this dimension nearly matches full-width quality — the rule for picking the smallest good-enough vector.
    return ((baseline['mrr'] - metrics['mrr']) <= TOLERANCE
            and (baseline[f'recall@{K}'] - metrics[f'recall@{K}']) <= TOLERANCE)


# Smallest dimension that still clears the quality bar; fall back to full width if none qualify.
eligible = [d for d in sorted(candidate_dims) if within_tolerance(dim_metrics[d])]
selected_dims = eligible[0] if eligible else baseline_dims
selected = dim_metrics[selected_dims]
rejected = [d for d in candidate_dims if d != selected_dims]

saved_pct = 1 - selected_dims / baseline_dims
backend_label = f'hosted {embedding_backend["deployment"]}' if embedding_backend['available'] else 'OFFLINE FALLBACK (not meaningful)'

print(f'Backend: {backend_label}')
print(f'Selected dimension: {selected_dims} (normalize={normalize})')
print(f"  recall@1 / recall@{K} / mrr: {selected['recall@1']:.0%} / {selected[f'recall@{K}']:.0%} / {selected['mrr']:.3f}")
print(f"  vs full {baseline_dims}-d baseline: {baseline['recall@1']:.0%} / {baseline[f'recall@{K}']:.0%} / {baseline['mrr']:.3f}")
print(f'  storage vs baseline: {saved_pct:.0%} smaller per vector')
print('\nRejected:')
for dims in rejected:
    m = dim_metrics[dims]
    gap_mrr = baseline['mrr'] - m['mrr']
    reason = 'quality gap > tolerance' if (gap_mrr > TOLERANCE or (baseline[f'recall@{K}'] - m[f'recall@{K}']) > TOLERANCE) else 'larger than needed'
    print(f"  {dims}-d: recall@{K}={m[f'recall@{K}']:.0%} mrr={m['mrr']:.3f} (Δmrr={gap_mrr:+.3f}) — {reason}")

if not embedding_backend['available']:
    print('\nWARNING: selection ran on the offline fallback. Re-run with Azure before trusting it.')


## 6. Export the Handoff

- **What gets exported:**
  - `.artifacts/embeddings.json` — one record per chunk (`chunk_id`, `work_id`, and the selected-dimension unit vector), the input notebook 04 loads to build the ANN index.
  - `.artifacts/embedding-handoff.json` — the decision record: backend and deployment, native and selected dimensions, the normalization decision, the tolerance used, the full candidate comparison, token/latency spend, and a back-reference to the chunking strategy.
- **Provenance preserved:** each vector keeps its `chunk_id`, so indexing, retrieval, and labels downstream still resolve back through the chunk to the original source lines.
- **Cache stays local:** the per-vector cache under `.artifacts/embedding-cache/` lets re-runs and future retuning skip the API entirely; it is ignored, like all of `.artifacts/`.
- **Hygiene:** `.artifacts/` is gitignored and notebook outputs are stripped on commit — do not commit vectors or the decision record.
- **Expected result:** both artifacts written, with the selected dimension, backend, and chunk count printed.


In [ ]:
selected_reps = chunk_reps[selected_dims]  # the chunk vectors at the chosen dimension (already normalized)
embeddings_path = ARTIFACTS / 'embeddings.json'
embedding_handoff_path = ARTIFACTS / 'embedding-handoff.json'

# embeddings.json: one record per chunk, carrying the chunk_id so notebook 04 can join back to provenance.
payload = {
    'dims': selected_dims,
    'normalized': normalize,
    'backend': 'hosted' if embedding_backend['available'] else 'offline-fallback',
    'deployment': embedding_backend['deployment'],
    'vectors': [
        {'chunk_id': chunks[i]['chunk_id'], 'work_id': chunks[i]['work_id'], 'embedding': vector}
        for i, vector in enumerate(selected_reps)
    ],
}
embeddings_path.write_text(json.dumps(payload), encoding='utf-8')  # compact: vectors are large

# embedding-handoff.json: the decision record and the measured comparison that justified it.
embedding_handoff = {
    'stage': '03-embeddings',
    'backend': payload['backend'],
    'deployment': embedding_backend['deployment'],
    'native_dims': native_dims,
    'selected_dims': selected_dims,
    'normalized': normalize,
    'tolerance': TOLERANCE,
    'candidates': dim_metrics,          # the measured recall/MRR across every candidate dimension
    'usage': usage_log,                 # tokens / API calls / latency spent embedding the corpus
    'chunk_count': len(chunks),
    'resolved_questions': len(gold),    # how many labeled questions the proxy could score
    'source_chunking_strategy': selected_strategy,
}
embedding_handoff_path.write_text(json.dumps(embedding_handoff, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'Wrote {len(selected_reps):,} {selected_dims}-d vectors to {embeddings_path.relative_to(ROOT)}')
print(f'Wrote decision record to {embedding_handoff_path.relative_to(ROOT)}')
print(f'Backend: {payload["backend"]}; cache retained at {CACHE_DIR.relative_to(ROOT)} for cost-free re-runs.')
if not embedding_backend['available']:
    print('Note: these vectors are the offline fallback — re-run against Azure to produce real embeddings.')


## Observations and Open Questions

Honest notes from working through this stage — what the numbers do and do not prove, and what to revisit later.

- **The retrieval proxy here is real nearest-neighbor search, but on a tiny sample.** Unlike notebook 02's metadata lookup, this notebook actually embeds questions and ranks chunks — that is a genuine signal. But it only covers the labeled dev questions that resolve to a parsed scene, so a one- or two-question swing moves the metrics a lot. Treat the dimension choice as a well-reasoned **default**, confirmed end-to-end in notebooks 05 (retrieval) and 10 (evaluation).

- **Dimension is the only model knob we could turn fairly.** With a single deployment we cannot compare `3-small` vs `3-large` or a non-OpenAI encoder without standing up more infrastructure. The Matryoshka truncation trick let us explore the quality/size tradeoff for *free* (one embedding pass), but it does **not** tell us whether a different model family would retrieve better. Comparing model families is deferred to when a second embedding deployment exists.

- **Truncating locally approximates the API's `dimensions` parameter — verify if it matters.** The 3-series models are trained so a renormalized prefix is a valid embedding, so local truncation and the server-side `dimensions` parameter should agree closely. We relied on that property rather than re-embedding at each width to save cost; if a borderline dimension decision ever hinges on it, re-embed that one candidate server-side and compare.

- **Normalization was the easy, safe call.** It changes nothing about ranking (confirmed), and it unlocks fast inner-product search downstream. There is no tradeoff to agonize over here — store unit vectors.

- **Caching is the quiet hero.** Embedding the whole corpus is the expensive part of this stage; the on-disk cache means re-running the notebook, retuning the tolerance, or changing the selection rule costs nothing. The cache key includes the deployment name, so switching models invalidates it automatically rather than silently serving stale vectors.

- **How we will revisit.** Embeddings are a swappable knob: change the selected dimension (or, later, the model), re-export `embeddings.json`, and re-run 04→05→08 on the **same** questions. Iterate on the **development** set; check the **held-out** set only periodically so we do not overfit the evaluation.


## Completion Check and Next Notebook

- **Restart and run in order:** restart the kernel and run top to bottom to confirm this notebook depends only on the notebook 02 artifacts, not hidden state. The first run calls the embedding API; later runs are served from the local cache.
- **What we achieved:** every chunk embedded once at full width with batching and caching; candidate dimensions (256 / 512 / 1024 / full) compared on a real retrieval proxy, storage, and cost; a normalization decision confirmed by a dot-vs-cosine check; a smallest-within-tolerance dimension selected; and two exported artifacts (`embeddings.json`, `embedding-handoff.json`).
- **Honest limits:** the retrieval proxy uses only the labeled dev questions that resolve to a parsed scene, so the sample is small; the real judges are retrieval (notebook 05) and continuous evaluation (notebook 10). Model-family comparison is deferred until a second embedding deployment exists.
- **Privacy and hygiene:** outputs are stripped on commit by `nbstripout`; `.artifacts/` (including the embedding cache) is ignored. Do not commit vectors or the decision record.
- **Next stage — 04 Indexing:** load `.artifacts/embeddings.json`, build an approximate-nearest-neighbor index (index type, ANN parameters, metadata schema) over the selected vectors, and measure build time and search latency/recall while holding the embedding selection fixed.


## Questions, Discussions, and Further Insights

The "Observations and Open Questions" cell records caveats about *our* measurements. These questions build the broader intuition.

**Q1. Why does normalization let us use dot product instead of cosine — and why care?**
Cosine similarity divides the dot product by both vectors' lengths to measure only *angle*. If every vector already has length 1, those divisions are by 1, so dot product *equals* cosine. We care because the next stage's index (04) runs fastest with plain inner-product math; by storing unit vectors we get cosine-quality ranking at inner-product speed, with **zero change to which passages rank first** (the notebook verifies this with the `norm_top == raw_top` check).

**Q2. Is truncating a vector to fewer dimensions really safe?**
For Matryoshka-trained models (OpenAI 3-series), yes — they are trained so a renormalized prefix is itself a valid embedding. That's what makes the dimension sweep free: embed once at full width, then slice. The caveat the notebook flags honestly: local truncation *approximates* the server-side `dimensions` parameter; if a borderline decision ever hinges on it, re-embed that one candidate server-side to confirm. For a non-Matryoshka model, truncation is **not** safe and you must request each dimension from the model.

**Q3. Why is "which model" the biggest lever yet the one we don't test?**
Because a stronger or domain-tuned embedding model can lift retrieval more than any dimension tweak — it raises the quality ceiling itself. We don't test it only because we have a single deployment, and a fair comparison means standing up each model, re-embedding the whole corpus, and re-running the identical retrieval proxy. It's deferred, not dismissed: when a second deployment exists, model choice should be the *first* thing re-measured.

**Q4. The retrieval proxy here embeds questions and ranks chunks — isn't that already retrieval?**
It's a *preview* of retrieval, run brute-force over all chunks with no index, on a tiny labeled sample. It's a genuine signal (unlike notebook 02's metadata-only proxy), but it's not the real system: there's no ANN index (04), no hybrid/keyword search or reranking (05–06), and the sample is small. It pre-screens the dimension; notebooks 05 and 10 are the end-to-end judges.

**Q5. Why cache embeddings, and why include the model name in the cache key?**
Embedding the corpus is the expensive part of this stage. Caching each vector to disk makes the first run cheap and every subsequent run (re-tuning the tolerance, changing the selection rule) essentially free. The cache key includes the deployment name so that switching models **invalidates** the cache automatically rather than silently serving vectors from the wrong model — a subtle but important correctness guard.

**Q6. Dense embeddings miss exact quotes and rare names — is that a problem for a Shakespeare quote desk?**
It can be, and the notebook names the fix without implementing it here: **hybrid search** (dense + keyword/BM25) in notebook 05. Dense embeddings are great at meaning and paraphrase but can under-weight an exact string like a specific character name or line. Fusing a sparse keyword signal recovers that precision. Embedding is deliberately the dense half only; retrieval adds the sparse half.

**Q7. What did we produce, and what happens next?**
We exported `.artifacts/embeddings.json` (one selected-dimension unit vector per chunk, keyed by `chunk_id`) and `.artifacts/embedding-handoff.json` (the decision record). Notebook 04 (Indexing) loads the vectors and builds a searchable approximate-nearest-neighbor index over them, holding this embedding selection fixed.
